# PolicyLens - evaluation report

Reproduces the tables and plots of the latest evaluation run (`experiments/latest.json`).

To create a new run first (backend stopped): `venv\Scripts\python ml\run_all.py`.

To open this notebook: `venv\Scripts\python -m pip install notebook` then `venv\Scripts\python -m notebook notebooks/evaluation_report.ipynb` (or open it in VS Code with the `venv` interpreter).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'experiments').exists() else Path.cwd().parent
latest = json.loads((ROOT / 'experiments' / 'latest.json').read_text())
RUN = ROOT / latest['path']
metrics = json.loads((RUN / 'metrics.json').read_text(encoding='utf-8'))
print('Run:', metrics['run'], '| updated', metrics['updated_at'])
pd.Series(metrics['dataset'])

## Headline metrics

In [ ]:
pd.Series(metrics['headline'], name='value').to_frame()

## Retrieval ablation

Share of answerable questions whose supporting clause is found at rank 1/3/5, and mean reciprocal rank.

In [ ]:
modes = ['bm25', 'dense', 'hybrid', 'hybrid_rerank']
retrieval = pd.DataFrame({m: metrics['retrieval'][m] for m in modes}).T
retrieval[['hit@1', 'hit@3', 'hit@5', 'mrr@10', 'recall@5']].round(3)

In [ ]:
colors = {'bm25': '#eb6834', 'dense': '#2a78d6', 'hybrid': '#e87ba4', 'hybrid_rerank': '#0d9488'}
ax = retrieval[['hit@1', 'hit@3', 'hit@5', 'mrr@10']].T.mul(100).plot.bar(
    color=[colors[m] for m in retrieval.index], figsize=(8, 4), rot=0, width=0.8)
ax.set_ylabel('% of questions'); ax.set_ylim(0, 105); ax.legend(frameon=False, ncol=4)
ax.spines[['top', 'right']].set_visible(False); plt.tight_layout()

## Answers

In [ ]:
a = metrics.get('answers', {})
summary = {k: a.get(k) for k in ('n', 'answerable', 'answer_accuracy', 'key_fact_recall', 'citation_accuracy')}
summary.update({'faithfulness_mean': a['faithfulness']['mean'], 'faithfulness_median': a['faithfulness']['median'],
                'abstention_accuracy': a['abstention']['accuracy'], 'false_abstentions': a['abstention']['false_abstentions'],
                'p50_ms': a['latency_ms'].get('p50_ms'), 'p95_ms': a['latency_ms'].get('p95_ms')})
pd.Series(summary).to_frame('value')

In [ ]:
pd.DataFrame(a['by_category']).T[['n', 'correct', 'accuracy']]

In [ ]:
answers = pd.read_json(RUN / 'answers.jsonl', lines=True)
answers[['id', 'language', 'status', 'key_fact_recall', 'citation_correct', 'faithfulness', 'model']].head(70)

## Claim Copilot

In [ ]:
c = metrics.get('claims', {})
print('accuracy:', c.get('accuracy'), '| faithfulness mean:', c.get('faithfulness_mean'))
pd.DataFrame(c['confusion_matrix'], index=[f'expected {l}' for l in c['labels']], columns=[f'predicted {l}' for l in c['labels']])

## Policy Card extraction

In [ ]:
e = metrics['extraction']
print('accuracy:', e['accuracy'], '| verified quotes:', e['verified_ratio'], '| models:', e['models'])
pd.Series(e['by_field'], name='accuracy').sort_values().to_frame()

## Saved plots

In [ ]:
from IPython.display import Image, display
for png in sorted((RUN / 'plots').glob('*.png')):
    print(png.name)
    display(Image(filename=str(png), width=640))